In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [8]:
# 모델을 사용하여 테스트 케이스에 대한 클로드의 답변 을 평가하는 채점 로직
def grade_by_model(test_case, output):
    eval_prompt = f"""
당신은 숙련된 AWS 코드 리뷰어입니다. 아래 AI가 생성한 솔루션을 평가해주세요.

원본 과제:
<task>
{test_case["task"]}
</task>

평가 대상 솔루션:
<solution>
{output}
</solution>

출력 형식
다음 필드를 포함한 구조화된 JSON 객체로 평가 결과를 작성하세요. 필드 순서는 아래와 같이 유지합니다:
- "strengths": 핵심 강점 1~3개를 담은 배열
- "weaknesses": 개선이 필요한 영역 1~3개를 담은 배열
- "reasoning": 전반적인 평가에 대한 간결한 설명
- "score": 1~10 사이의 숫자

JSON 형식으로만 응답하세요. 답변은 간결하고 직접적으로 작성합니다.

응답 예시 구조:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """
    messages = []
    add_user_message(messages, eval_prompt)
    # haiku 모델은 여전히 message_prefilling을 지원하므로 강의 예시대로 이를 사용함
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [25]:
import json

# 클로드 개별 업무 지시용 함수
def run_prompt(test_case):
    """프롬프트와 테스트 케이스 입력값을 병합한 후 결과를 반환"""
    prompt = f"""
    다음 작업을 수행하라:

    {test_case["task"]}
    """

    messages = []
    add_user_message(messages, prompt)
    # 맥스 토큰 내로 점수를 계산하기 위한 간결화 system prompt
    output = chat(messages, system="핵심 코드만 간결하게 출력해라. 설명, 주석, 대안 없이.")
    return output

# 클로드 개별 항목 검토 및 채점용 함수
def run_test_case(test_case):
    """run_prompt를 호출한 후 결과를 채점"""
    # 문제 먼저 풀고
    output = run_prompt(test_case)

    # 채점로직_ 그 답을 검토시키고
    model_grade= grade_by_model(test_case, output)
    # 검토 결과에서 점수만 추출
    score = model_grade["score"]
    # 검토 결과에서 설명만 추출
    reasoning = model_grade["reasoning"]
    

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

# 파이썬에서 평균 계산기 호출

from statistics import mean

# 위 함수를 활용하여 수행한 작업 및 이에 대한 채점 결과를 순차적으로 제시
def run_eval(dataset):
    """데이터셋을 로드하고 각 테스트 케이스에 run_test_case를 호출"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    # 모든 테스트 케이스의 점수만 뽑아내어 이를 평균하여 제시
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    return results

with open("dataset.json", "r") as f:
    dataset = json.load(f)



In [26]:
results = run_eval(dataset)
# 결과값을 unicode가 아닌 한글로 인출하기 위해 ensure_ascii=False를 추가
json.dumps(results, indent=2, ensure_ascii=False)

Average score: 5.666666666666667


'[\n  {\n    "output": "```python\\ndef extract_s3_bucket(s3_uri):\\n    return s3_uri.replace(\\"s3://\\", \\"\\").split(\\"/\\")[0]\\n\\n# 사용 예\\nprint(extract_s3_bucket(\\"s3://my-bucket/path/to/file.txt\\"))  # my-bucket\\n```\\n\\n또는 정규식을 사용한 방법:\\n\\n```python\\nimport re\\n\\ndef extract_s3_bucket(s3_uri):\\n    match = re.match(r\\"s3://([^/]+)\\", s3_uri)\\n    return match.group(1) if match else None\\n\\n# 사용 예\\nprint(extract_s3_bucket(\\"s3://my-bucket/path/to/file.txt\\"))  # my-bucket\\n```\\n\\n또는 urllib를 사용한 방법:\\n\\n```python\\nfrom urllib.parse import urlparse\\n\\ndef extract_s3_bucket(s3_uri):\\n    return urlparse(s3_uri).netloc\\n\\n# 사용 예\\nprint(extract_s3_bucket(\\"s3://my-bucket/path/to/file.txt\\"))  # my-bucket\\n```",\n    "test_case": {\n      "task": "Parse an AWS S3 bucket name from an S3 URI (e.g., s3://my-bucket/path/to/file.txt) and extract just the bucket name",\n      "format": "regex"\n    },\n    "score": 6,\n    "reasoning": "솔루션은 기본적인 기능을 수행하지만